# Phase 1 EDA — paper Section 2

Renders the figures and tables produced by `scripts/01_audit_filter.py`.

Per spec detail §0.1 this notebook **imports from `src/` and defines no analysis logic inline** — it exists to render. If a number here looks wrong, fix it in `src/lyra_capstone/data/` and re-run the script.

**Prerequisite:** `python scripts/00_download.py && python scripts/01_audit_filter.py`

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from IPython.display import Image, Markdown, display

from lyra_capstone import paths

report = json.loads((paths.INTERIM / "audit_report.json").read_text())
filtered = pd.read_parquet(paths.INTERIM / "filtered.parquet")
corpus = pd.read_parquet(paths.RAW_MEDEMBED / "corpus.parquet")
print(f"filtered: {len(filtered):,} queries | corpus: {len(corpus):,} passages")

## Table 1 — Dataset after filtering

The paper must report the **unique-query** count, not the 232,684 triplet rows (spec v2 §2.1).

In [ ]:
raw_q = report["input"]["queries"]
rows = [
    ("Unique queries (raw)", raw_q, ""),
    ("  \u2212 anaphoric (D5)", -report["anaphoric"]["n_flagged_anaphoric"],
     f"{100 * report['anaphoric']['flagged_rate']:.2f}%"),
    ("  \u2212 degenerate positives", -report["degenerate_positives"].get("n_removed", 0),
     f"{100 * report['degenerate_positives'].get('removal_rate', 0):.2f}%"),
    ("**Surviving queries**", report["surviving_queries"],
     f"{100 * report['surviving_rate_vs_raw']:.1f}% of raw"),
    ("Corpus passages (unchanged)", len(corpus), "full index at eval"),
    ("Mean negatives / query", report["mean_negatives_per_query"], ""),
]
display(pd.DataFrame(rows, columns=["Quantity", "Value", "Note"]))

## Figure 1 — Length distributions

The passage-token tail is what justifies `max_seq_length = 320` in Experiment A.

In [ ]:
display(Image(filename=str(paths.FIGURES / "eda_lengths.png")))

## Figure 2 — Anaphoric query filter (D5)

The middle bar is the filter's second clause at work: queries anaphoric *in form* that name a clinical entity and so remain retrievable.

In [ ]:
display(Image(filename=str(paths.FIGURES / "eda_anaphoric.png")))

a = report["anaphoric"]
display(pd.DataFrame([
    ("Queries with a definite patient reference", a["n_with_definite_reference"]),
    ("  of which saved by the clinical-term clause", a["n_saved_by_clinical_term_clause"]),
    ("  of which flagged anaphoric and dropped", a["n_flagged_anaphoric"]),
    ("Queries naming \u22651 clinical entity", f"{100 * a['queries_with_clinical_term_rate']:.1f}%"),
], columns=["Quantity", "Value"]))

if a.get("hand_validation"):
    display(Markdown("### Filter validation against hand labels"))
    display(pd.DataFrame([a["hand_validation"]]).T.rename(columns={0: "value"}))
else:
    display(Markdown(
        "> **Hand validation pending.** Label "
        "`artifacts/interim/filter_validation_sample.csv` "
        "(`human_label`: 1 = anaphoric, 0 = answerable), then re-run "
        "`01_audit_filter.py --validation-labels <file>`. "
        "An unvalidated filter that removes 10% of the dataset will not survive review."
    ))

## Figure 3 — False negatives and generic filler (Issue 2)

The reuse tail is the headline here: single boilerplate passages serve hundreds of unrelated queries as negatives.

In [ ]:
display(Image(filename=str(paths.FIGURES / "eda_negatives.png")))

fn = report.get("false_negatives", {})
if "distribution" in fn:
    d = fn["distribution"]
    display(pd.DataFrame(
        [(f"p{k}", round(v, 4)) for k, v in d["percentiles"].items()],
        columns=["percentile", "cosine"],
    ).T)
if fn.get("applied"):
    display(pd.DataFrame([fn["stats"]]).T.rename(columns={0: "value"}))

# Most-reused negatives — the concrete examples for the Section 2 narrative.
scores = paths.INTERIM / "negative_scores.parquet"
if scores.exists():
    pairs = pd.read_parquet(scores)
    text = dict(zip(corpus["id"], corpus["text"]))
    reuse = pairs.groupby("neg_id")["query_id"].nunique().sort_values(ascending=False)
    display(Markdown("### Most-reused negatives"))
    display(pd.DataFrame(
        [(n, text[i][:110]) for i, n in reuse.head(8).items()],
        columns=["distinct queries served", "passage"],
    ))

## Figure 4 — Query typology

Keyword-style vs. natural-question. Both styles must be represented in the test slice.

In [ ]:
display(Image(filename=str(paths.FIGURES / "eda_query_types.png")))

## Figure 5 — Lexical overlap

How much of the task is solvable by exact matching. This is the quantitative motivation for carrying BM25 as configuration 1 — and for the spec's warning that BM25 may beat stock bge-m3.

In [ ]:
display(Image(filename=str(paths.FIGURES / "eda_lexical_overlap.png")))

## Figure 6 — Corpus topical coverage

Specialty / body-system breadth, by MeSH disease category.

In [ ]:
display(Image(filename=str(paths.FIGURES / "eda_specialty.png")))

s = report["specialty"]
print(f"{s['n_passages_scored']:,} passages scored; "
      f"{s['n_no_mesh_disease_match']:,} with no MeSH disease match")
display(pd.DataFrame(list(s["counts"].items())[:15], columns=["MeSH category", "passages"]))

## Provenance

Every number above traces to an artifact manifest (Definition of Done).

In [ ]:
for name in ("raw", "interim"):
    m = json.loads((paths.ARTIFACTS / name / "manifest.json").read_text())
    print(f"{name:8s} phase={m['phase']:<18s} git={m['git_sha'][:8]} "
          f"dirty={m['git_dirty']} seed={m['seed']} {m['timestamp_utc']}")